In [11]:
import pandas as pd
import numpy as np

# 1. Carga de datos [cite: 43]
df = pd.read_csv('creditcard.csv')

# 2. Preparación de Dimensiones (Transformación de datos)
# Convertir segundos a horas y luego a categorías
df['Hora'] = (df['Time'] // 3600) % 24
df['Time_Bin'] = pd.cut(df['Hora'], bins=[0, 6, 12, 18, 24],
                        labels=['Madrugada', 'Mañana', 'Tarde', 'Noche'], include_lowest=True)

# Categorizar el Importe en rangos
df['Amount_Range'] = pd.cut(df['Amount'], bins=[0, 50, 500, df['Amount'].max()],
                            labels=['Bajo', 'Medio', 'Alto'], include_lowest=True)

# 3. Construcción del Cubo de Datos (Pivot Table) [cite: 50, 56]
cubo = pd.pivot_table(df,
                      values='Amount',
                      index=['Class', 'Time_Bin'],
                      columns=['Amount_Range'],
                      aggfunc=np.sum,
                      fill_value=0)

print("--- Cubo de Datos Original ---")
print(cubo)

# Construcción del cubo usando el PROMEDIO
cubo_promedio = pd.pivot_table(df,
                               values='Amount',
                               index=['Class', 'Time_Bin'],
                               columns=['Amount_Range'],
                               aggfunc=np.mean, # Cambiamos a promedio
                               fill_value=0)

print("--- Cubo de Datos (Promedios) ---")
print(cubo_promedio)

--- Cubo de Datos Original ---
Amount_Range          Bajo       Medio        Alto
Class Time_Bin                                    
0     Madrugada  288577.51   899396.90   532920.28
      Mañana     793864.77  4213717.37  3311116.77
      Tarde      934741.83  4718752.79  3847785.90
      Noche      727230.19  2941712.03  1892645.70
1     Madrugada     589.65     4273.17     6960.65
      Mañana        587.45     7603.37     8580.24
      Tarde         492.64     8630.76    13170.38
      Noche         404.19     6351.84     2483.63
--- Cubo de Datos (Promedios) ---
Amount_Range          Bajo       Medio         Alto
Class Time_Bin                                     
0     Madrugada  13.547604  147.611505  1049.055669
      Mañana     15.513958  151.213571  1079.242754
      Tarde      14.676200  154.348842  1056.793711
      Noche      13.325091  147.542985  1001.399841
1     Madrugada   5.896500  170.926800   870.081250
      Mañana      9.475000  138.243091   953.360000
      Tar

/tmp/ipython-input-194/216497234.py:18: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  cubo = pd.pivot_table(df,
/tmp/ipython-input-194/216497234.py:18: FutureWarning: The provided callable <function sum at 0x798bf8dcc9a0> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  cubo = pd.pivot_table(df,
/tmp/ipython-input-194/216497234.py:29: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  cubo_promedio = pd.pivot_table(df,
/tmp/ipython-input-194/216497234.py:29: FutureWarning: The provided callable <function mean at 0x798bf8dcda80> is currently usin

In [9]:
# Rebanar el cubo para ver solo transacciones fraudulentas
# 4. Operación SLICE (Rebanar por Fraude) [cite: 60, 62]
slice_fraude = cubo.loc[1]
print("\nResultado de SLICE (Solo Fraudes):")
print(slice_fraude)


Resultado de SLICE (Solo Fraudes):
Rango_Importe   Pequeño   Mediano   Grande
Franja_Horaria                            
Madrugada       1210.41   7584.49  3028.57
Mañana          3516.46   9793.73  3460.87
Tarde           1378.80  14167.09  6747.89
Noche            913.59   8326.07     0.00


In [8]:
# Agregamos los datos eliminando 'Franja_Horaria'
# 5. Operación ROLL-UP (Agregación eliminando el tiempo) [cite: 68, 69]

rollup_resultado = df.groupby('Class')['Amount'].sum()
print("\nResultado de ROLL-UP (Total por Clase):")
print(rollup_resultado)


Resultado de ROLL-UP (Total por Clase):
Class
0    25102462.04
1       60127.97
Name: Amount, dtype: float64
